In [ ]:
import numpy as np
import open3d as o3d
from plyfile import PlyData, PlyElement   
from pt_cloud_utils import *
import copy

In [ ]:
# go from ply to txt
front_ply = PlyData.read('/Users/adeleyounis/Desktop/Capstone/wAI/point-clouds/front.ply')  # add file name
front_data = np.array([list(x) for x in front_ply.elements[0].data])
np.savetxt('/Users/adeleyounis/Desktop/Capstone/wAI/point-clouds/front.txt', front_data)

side_ply = PlyData.read('/Users/adeleyounis/Desktop/Capstone/wAI/point-clouds/side.ply')  # add file name
side_data = np.array([list(x) for x in side_ply.elements[0].data])
np.savetxt('/Users/adeleyounis/Desktop/Capstone/wAI/point-clouds/side.txt', side_data)

Import Point Cloud txt file

In [ ]:
# might change based on what we are actually using 
front_pt_cloud = np.loadtxt("/Users/adeleyounis/Desktop/Capstone/wAI/point-clouds/front.txt", delimiter=' ')
points_front = front_pt_cloud[:,:3]
colors_front = front_pt_cloud[:,3:]


side_pt_cloud = np.loadtxt("/Users/adeleyounis/Desktop/Capstone/wAI/point-clouds/side.txt", delimiter=' ')
points_side = side_pt_cloud[:,:3]
colors_side = side_pt_cloud[:,3:]



In [ ]:
# generate a mesh
# make empty point cloud object for front and side views
pcd_front = o3d.geometry.PointCloud()
pcd_side = o3d.geometry.PointCloud()

# add points to the point cloud object
pcd_front.points = o3d.utility.Vector3dVector(points_front)
pcd_side.points = o3d.utility.Vector3dVector(points_side)

# downsample so faster
downpcd_front = pcd_front.voxel_down_sample(voxel_size=0.05)
downpcd_side = pcd_side.voxel_down_sample(voxel_size=0.05)

downpcd_front.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
    radius=0.05, max_nn=30))

downpcd_front.orient_normals_to_align_with_direction()

downpcd_side.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(
    radius=0.05, max_nn=30))

# downpcd_side.orient_normals_to_align_with_direction()



In [ ]:
rotated_downpcd_side = copy.deepcopy(downpcd_side)

downpcd_front.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)
downpcd_side.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30)
)

# rotate about the center of the point cloud
R_y = rotated_downpcd_side.get_rotation_matrix_from_xyz((0, -np.pi/2, 0))
center = rotated_downpcd_side.get_center()
rotated_downpcd_side.rotate(R_y, center=center)
side = stretch_vertically(rotated_downpcd_side, scale_factor=1.2)

# make another side view, where the front is sandwiched between two sides
# copy side
front_width = get_cloud_width(downpcd_front)

downpcd_side = copy.deepcopy(side)
downpcd_side.translate((-front_width/2, -30 , 0))

downpcd_side2 = copy.deepcopy(side)
downpcd_side2.translate((front_width/2, -30, 0))


threshold = 0.05
trans_init = np.eye(4)
reg_icp = o3d.pipelines.registration.registration_icp(
    downpcd_side, downpcd_front, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
)

T = reg_icp.transformation

downpcd_side.transform(T)
downpcd_side2.transform(T)

o3d.visualization.draw_geometries([downpcd_front, downpcd_side, downpcd_side2])

In [ ]:
# combine pcds
# side view is facing left, front view is facing forward
# want side view 90 degrees to the left

pcd_front_t = downpcd_front
pcd_side_t = downpcd_side

R = pcd_side_t.get_rotation_matrix_from_xyz((0, np.pi/2, 0))
pcd_side_t.rotate(R, center=(0, 0, 0))
# pcd_side_t.translate((0.7, 0, 0))  # adjust based on object size/position

combined_pcd = pcd_front_t + pcd_side_t

# Optional: visualize combined point cloud before meshing
o3d.visualization.draw_geometries([combined_pcd])



In [ ]:
# get downsamples pts
points = np.asarray(downpcd_side.points)
normals = np.asarray(downpcd_side.normals)

inflate_amount = 10  # thicken up
points_inflated = points + normals * inflate_amount

# merge original + inflated
points_all = np.vstack([points, points_inflated])
pcd_thick = o3d.geometry.PointCloud()
pcd_thick.points = o3d.utility.Vector3dVector(points_all)

pcd_thick.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.05, max_nn=30
    )
)


In [ ]:
num_layers = 20
max_offset = -20 # figure out what this is a measurement of
all_layers = []

points = np.asarray(pcd_thick.points)
normals = np.asarray(pcd_thick.normals)

for i in range(num_layers):
    t = i / (num_layers - 1)
    weight = np.exp(-((t-1)**2) / 0.3)  
    offset = max_offset * weight
    points_layer = points + normals * offset
    all_layers.append(points_layer)

points_all = np.vstack(all_layers)
pcd_soft = o3d.geometry.PointCloud()
pcd_soft.points = o3d.utility.Vector3dVector(points_all)
# down sample for faster 
pcd_soft = pcd_soft.voxel_down_sample(voxel_size=0.005)

pcd_soft.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.05, max_nn=30
    )
)

o3d.visualization.draw_geometries([pcd_soft])

In [ ]:
# possion mesh - best for volume since it fills all holes
# pros: produces smooth, tight meshes, good for incomplete/noisy data.
# cons: slow, needs accurate normals, can over-smooth details.
tri_mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_soft, depth=9)
tri_mesh.compute_vertex_normals()
tri_mesh.paint_uniform_color([0.1, 0.8, 0.2]) # green
o3d.visualization.draw_geometries([tri_mesh])

In [ ]:
# ball pivoting mesh - rolls "virtual ball" over point cloud to form triangles
# pros: fast, good for dense, clean data, preserves sharp features.
# cons: needs well-sampled data, can miss thin structures.

radii = [3, 4.5, 5] # larger == faster, rougher
ball_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    pcd_soft, o3d.utility.DoubleVector(radii))
ball_mesh.compute_vertex_normals()
ball_mesh.paint_uniform_color([0.8, 0.1, 0.2]) # red
o3d.visualization.draw_geometries([ball_mesh])

In [ ]:
# alpha shapes mesh - generalizes convex hull, can capture concavities
# pros: captures concavities, good for varied shapes.
# cons: sensitive to alpha parameter, can produce holes or disconnected components.

alpha = 0.1  # smaller = tighter to data, larger = smoother
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd_soft, alpha)
mesh.compute_vertex_normals()
o3d.visualization.draw_geometries([mesh])

In [ ]:
# convex hull - smallest convex shape enclosing all points
hull, _ = pcd_soft.compute_convex_hull()
hull_ls = o3d.geometry.LineSet.create_from_triangle_mesh(hull)
o3d.visualization.draw_geometries([hull_ls])

Volume

In [ ]:
# create voxel grid from point cloud
voxel = 1
voxel_grid_cld=o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_soft, voxel_size=voxel) 
print(voxel_grid_cld.get_voxels())

In [ ]:
# Get voxel coordinates
voxels = np.asarray([v.grid_index for v in voxel_grid_cld.get_voxels()])

# Create a 3D occupancy array
max_idx = voxels.max(axis=0) + 1
volume = np.zeros(max_idx, dtype=np.uint8)
volume[voxels[:,0], voxels[:,1], voxels[:,2]] = 1

print("Volume shape:", volume.shape)


In [ ]:
num_voxels = len(voxel_grid_cld.get_voxels())
volume_est = num_voxels * (voxel ** 3)

print(f"Occupied voxels: {num_voxels}")
print(f"Estimated volume: {volume_est} mm^3")


In [ ]:
# create visualizer window
vis = o3d.visualization.Visualizer()
vis.create_window(window_name='wAI Visualize', width=800, height=600)

vis.add_geometry(voxel_grid_cld)
vis.run()
vis.destroy_window()

In [ ]:
# get volume right from mesh
voxel_size = 1
voxel_grid_mesh = o3d.geometry.VoxelGrid.create_from_triangle_mesh(tri_mesh, voxel_size=voxel_size)

voxels = np.asarray([v.grid_index for v in voxel_grid_mesh.get_voxels()])

max_idx = voxels.max(axis=0) + 1
volume = np.zeros(max_idx, dtype=np.uint8)
volume[voxels[:,0], voxels[:,1], voxels[:,2]] = 1

print("Volume shape:", volume.shape)

num_voxels = len(voxel_grid_mesh.get_voxels())
volume_est = num_voxels * (voxel_size ** 3)

print(f"Occupied voxels: {num_voxels}")
print(f"Estimated volume: {volume_est} m^3")

# create visualizer window
vis = o3d.visualization.Visualizer()
vis.create_window(window_name='wAI Visualize', width=800, height=600)

vis.add_geometry(voxel_grid_mesh)
vis.run()
vis.destroy_window()